In [1]:
# Project paths and source verification

# Set up project paths and verify the original dataset

from pathlib import Path
import hashlib
import json
import shutil
import sys
import warnings

import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\Users\hp\Nassau_Candy_Final_Submission")

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "Nassau Candy Distributor.csv"
PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "Nassau_Candy_Feature_Engineered.csv"

APP_ROOT = PROJECT_ROOT / "data" / "processed" / "app_bundle"
APP_DATA = APP_ROOT / "data"
APP_MODELS = APP_ROOT / "models"
APP_REPORTS = APP_ROOT / "reports"
APP_SRC = APP_ROOT / "src"

for folder in [APP_DATA, APP_MODELS, APP_REPORTS, APP_SRC]:
    folder.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

EXPECTED_SOURCE_HASH = (
    "6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f"
)

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Original dataset not found: {RAW_FILE}")

if not PROCESSED_FILE.exists():
    raise FileNotFoundError(f"Processed dataset not found: {PROCESSED_FILE}")

source_hash_before = sha256_file(RAW_FILE)

if source_hash_before != EXPECTED_SOURCE_HASH:
    raise ValueError(
        "Original CSV SHA-256 does not match the previously verified hash. "
        "Stop and investigate before continuing."
    )

raw_df = pd.read_csv(RAW_FILE)
processed_df = pd.read_csv(PROCESSED_FILE)

if raw_df.shape != (10194, 18):
    raise ValueError(f"Unexpected original dataset shape: {raw_df.shape}")

print("Project setup verified.")
print("Original dataset:", raw_df.shape)
print("Processed dataset:", processed_df.shape)
print("Original SHA-256:", source_hash_before)
print("App output folder:", APP_ROOT)


Project setup verified.
Original dataset: (10194, 18)
Processed dataset: (10194, 34)
Original SHA-256: 6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f
App output folder: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle


In [3]:
# Processed dataset export
# Processed dataset ki verified working copy app ke data folder mein export hogi. Source file unchanged rahegi.


# Export the processed dataset for the application

import pandas as pd

if not PROCESSED_FILE.exists():
    raise FileNotFoundError(PROCESSED_FILE)

processed_export = pd.read_csv(PROCESSED_FILE)

if len(processed_export) != 10194:
    raise ValueError("Processed dataset row count does not match the verified project dataset.")

processed_output = APP_DATA / "processed_dataset.csv"
processed_export.to_csv(processed_output, index=False)

# Reload and verify the exported file
reloaded_processed = pd.read_csv(processed_output)

if not processed_export.equals(reloaded_processed):
    raise AssertionError("Processed dataset export verification failed.")

print("Processed dataset exported:", processed_output)
print("Export shape:", reloaded_processed.shape)
print("Original columns preserved:", set(raw_df.columns).issubset(reloaded_processed.columns))


Processed dataset exported: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\data\processed_dataset.csv
Export shape: (10194, 34)
Original columns preserved: True


In [11]:

# Export the verified  Factory Master

import pandas as pd
from pathlib import Path

# Verified source file from Phase 07
FACTORY_MASTER_SOURCE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "factory_coordinates.csv"
)

FACTORY_MASTER_OUTPUT = APP_DATA / "factory_master.csv"

# Check source availability
if not FACTORY_MASTER_SOURCE.exists():
    raise FileNotFoundError(
        f"Verified Factory Master not found: {FACTORY_MASTER_SOURCE}"
    )

# Load the verified factory master
factory_master_df = pd.read_csv(FACTORY_MASTER_SOURCE)

# Validate expected columns
required_columns = [
    "Factory_Name",
    "Latitude",
    "Longitude"
]

missing_columns = [
    col for col in required_columns
    if col not in factory_master_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Factory Master columns: {missing_columns}"
    )

# Validate records
if factory_master_df.empty:
    raise ValueError("Factory Master contains no records.")

if factory_master_df["Factory_Name"].isna().any():
    raise ValueError("Factory names contain missing values.")

if factory_master_df["Factory_Name"].duplicated().any():
    raise ValueError("Duplicate factory names found.")

# Validate coordinate data
factory_master_df["Latitude"] = pd.to_numeric(
    factory_master_df["Latitude"], errors="raise"
)

factory_master_df["Longitude"] = pd.to_numeric(
    factory_master_df["Longitude"], errors="raise"
)

if not factory_master_df["Latitude"].between(-90, 90).all():
    raise ValueError("Invalid latitude values found.")

if not factory_master_df["Longitude"].between(-180, 180).all():
    raise ValueError("Invalid longitude values found.")

if len(factory_master_df) != 5:
    raise ValueError(
        f"Expected 5 reference factories, found {len(factory_master_df)}."
    )

# Export app copy
factory_master_df.to_csv(
    FACTORY_MASTER_OUTPUT,
    index=False
)

# Reload and verify export
factory_master_check = pd.read_csv(FACTORY_MASTER_OUTPUT)

pd.testing.assert_frame_equal(
    factory_master_df,
    factory_master_check,
    check_dtype=False
)

print("=" * 65)
print("FACTORY MASTER EXPORT SUCCESSFUL")
print("=" * 65)

print("Source:", FACTORY_MASTER_SOURCE)
print("Output:", FACTORY_MASTER_OUTPUT)
print("Total factories:", len(factory_master_check))
print("Columns:", factory_master_check.columns.tolist())

print("\nFactory Master records:")
print(factory_master_check.to_string(index=False))

print("\nExport verification: PASSED")


FACTORY MASTER EXPORT SUCCESSFUL
Source: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\factory_coordinates.csv
Output: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\data\factory_master.csv
Total factories: 5
Columns: ['Factory_Name', 'Latitude', 'Longitude']

Factory Master records:
     Factory_Name  Latitude   Longitude
    Lot's O' Nuts 32.881893 -111.768036
  Wicked Choccy's 32.076176  -81.088371
      Sugar Shack 48.119140  -96.181150
   Secret Factory 41.446333  -90.565487
The Other Factory 35.117500  -89.971107

Export verification: PASSED


## Product-factory mapping export


In [25]:
# Product-F​​actory Mapping Export - independent and validated implementation

from pathlib import Path
import pandas as pd

# Define source and output paths
PROJECT_ROOT = Path(r"C:\Users\hp\Nassau_Candy_Final_Submission")

APP_ROOT = PROJECT_ROOT / "data" / "processed" / "app_bundle"
APP_DATA = APP_ROOT / "data"

MAPPING_SOURCE = (
    PROJECT_ROOT / "data" / "processed" / "product_factory_mapping.csv"
)

FACTORY_MASTER_FILE = APP_DATA / "factory_master.csv"

PROCESSED_SOURCE = (
    PROJECT_ROOT / "data" / "processed" /
    "Nassau_Candy_Feature_Engineered.csv"
)

MAPPING_OUTPUT = APP_DATA / "product_factory_mapping.csv"

# Check all source files
for source_file in [
    MAPPING_SOURCE,
    FACTORY_MASTER_FILE,
    PROCESSED_SOURCE
]:
    if not source_file.exists():
        raise FileNotFoundError(
            f"Required source file not found: {source_file}"
        )

# Load mapping and Factory Master independently
mapping_df = pd.read_csv(MAPPING_SOURCE)
factory_master_df = pd.read_csv(FACTORY_MASTER_FILE)
processed_df = pd.read_csv(PROCESSED_SOURCE)

# Validate mapping columns
required_mapping_columns = [
    "Product Name",
    "Mapped Factory",
    "Reference Division"
]

missing_columns = [
    col for col in required_mapping_columns
    if col not in mapping_df.columns
]

if missing_columns:
    raise ValueError(
        f"Mapping file is missing required columns: {missing_columns}"
    )

# Validate Factory Master columns
required_factory_columns = [
    "Factory_Name",
    "Latitude",
    "Longitude"
]

missing_factory_columns = [
    col for col in required_factory_columns
    if col not in factory_master_df.columns
]

if missing_factory_columns:
    raise ValueError(
        f"Factory Master is missing columns: {missing_factory_columns}"
    )

if "Product Name" not in processed_df.columns:
    raise ValueError(
        "Processed dataset does not contain Product Name."
    )

# Select only required mapping columns
mapping_df = mapping_df[required_mapping_columns].copy()

# Check empty data
if mapping_df.empty:
    raise ValueError("Product-factory mapping file is empty.")

# Check missing values
if mapping_df[required_mapping_columns].isna().any().any():
    raise ValueError("Mapping file contains missing required values.")

# Check duplicate product mappings
if mapping_df["Product Name"].duplicated().any():
    duplicates = mapping_df.loc[
        mapping_df["Product Name"].duplicated(keep=False),
        "Product Name"
    ].tolist()

    raise ValueError(
        f"Duplicate Product Name entries found: {duplicates}"
    )

# Validate mapped factories against exported Factory Master
known_factories = set(
    factory_master_df["Factory_Name"].astype(str).str.strip()
)

mapped_factories = set(
    mapping_df["Mapped Factory"].astype(str).str.strip()
)

unknown_factories = mapped_factories - known_factories

if unknown_factories:
    raise ValueError(
        f"Mapping contains factories absent from Factory Master: "
        f"{sorted(unknown_factories)}"
    )

# Validate complete product coverage
source_products = set(
    processed_df["Product Name"].dropna().astype(str).str.strip()
)

mapping_products = set(
    mapping_df["Product Name"].astype(str).str.strip()
)

unmapped_products = source_products - mapping_products
extra_products = mapping_products - source_products

if unmapped_products:
    raise ValueError(
        f"Products without factory mapping: {sorted(unmapped_products)}"
    )

if extra_products:
    raise ValueError(
        f"Mapping contains products absent from processed data: "
        f"{sorted(extra_products)}"
    )

if len(mapping_df) != len(source_products):
    raise ValueError(
        "Mapping row count does not match unique source product count."
    )

# Export mapping
APP_DATA.mkdir(parents=True, exist_ok=True)

mapping_df.to_csv(MAPPING_OUTPUT, index=False)

# Reload and verify export
mapping_check = pd.read_csv(MAPPING_OUTPUT)

pd.testing.assert_frame_equal(
    mapping_df.reset_index(drop=True),
    mapping_check.reset_index(drop=True),
    check_dtype=False
)

# Final result
print("=" * 65)
print("PRODUCT-FACTORY MAPPING EXPORT SUCCESSFUL")
print("=" * 65)

print("Source:", MAPPING_SOURCE)
print("Factory Master:", FACTORY_MASTER_FILE)
print("Output:", MAPPING_OUTPUT)

print("Mapped products:", len(mapping_check))
print("Referenced factories:", mapping_check["Mapped Factory"].nunique())
print("Unmapped products:", len(unmapped_products))
print("Unknown factories:", len(unknown_factories))

print("\nMapping records:")
print(mapping_check.to_string(index=False))

print("\nExport verification: PASSED")

PRODUCT-FACTORY MAPPING EXPORT SUCCESSFUL
Source: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\product_factory_mapping.csv
Factory Master: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\data\factory_master.csv
Output: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\data\product_factory_mapping.csv
Mapped products: 15
Referenced factories: 5
Unmapped products: 0
Unknown factories: 0

Mapping records:
                     Product Name    Mapped Factory Reference Division
Wonka Bar - Nutty Crunch Surprise     Lot's O' Nuts          Chocolate
        Wonka Bar - Fudge Mallows     Lot's O' Nuts          Chocolate
   Wonka Bar -Scrumdiddlyumptious     Lot's O' Nuts          Chocolate
       Wonka Bar - Milk Chocolate   Wicked Choccy's          Chocolate
Wonka Bar - Triple Dazzle Caramel   Wicked Choccy's          Chocolate
                      Laffy Taffy       Sugar Shack              Sugar
                        SweeTARTS       Sugar S

In [19]:
# Model Export

# Lead-Time Model Export - copy and reload the saved model

import json
import shutil
import joblib

MODEL_SOURCE = PROJECT_ROOT / "models" / "Nassau_Lead_Time_Model.joblib"

MODEL_METADATA_SOURCE = (
    PROJECT_ROOT / "models" / "Nassau_Lead_Time_Model_Metadata.json"
)

MODEL_OUTPUT = APP_MODELS / "Nassau_Lead_Time_Model.joblib"
MODEL_METADATA_OUTPUT = APP_MODELS / "model_metadata.json"
FEATURE_SCHEMA_OUTPUT = APP_MODELS / "feature_schema.json"

if not MODEL_SOURCE.exists():
    raise FileNotFoundError(f"Saved model not found: {MODEL_SOURCE}")

if not MODEL_METADATA_SOURCE.exists():
    raise FileNotFoundError(
        f"Model metadata not found: {MODEL_METADATA_SOURCE}"
    )

with open(MODEL_METADATA_SOURCE, "r", encoding="utf-8") as file:
    model_metadata = json.load(file)

feature_names = model_metadata.get("predictor_features", [])

if not feature_names:
    raise ValueError("Predictor feature list is missing from model metadata.")

if model_metadata.get("target_variable") != "Lead_Time_Days":
    raise ValueError("Unexpected model target variable.")

# Test source model before copying
source_model = joblib.load(MODEL_SOURCE)

shutil.copy2(MODEL_SOURCE, MODEL_OUTPUT)
shutil.copy2(MODEL_METADATA_SOURCE, MODEL_METADATA_OUTPUT)

feature_schema = {
    "target_variable": model_metadata["target_variable"],
    "predictor_features": feature_names,
    "feature_count": len(feature_names),
    "selected_model": model_metadata.get("selected_model")
}

with open(FEATURE_SCHEMA_OUTPUT, "w", encoding="utf-8") as file:
    json.dump(feature_schema, file, indent=4)

# Reload copied model
reloaded_model = joblib.load(MODEL_OUTPUT)

if reloaded_model is None:
    raise RuntimeError("Exported model could not be reloaded.")

print("=" * 65)
print("MODEL EXPORT SUCCESSFUL")
print("=" * 65)
print("Source:", MODEL_SOURCE)
print("Output:", MODEL_OUTPUT)
print("Model:", model_metadata.get("selected_model"))
print("Target:", model_metadata["target_variable"])
print("Predictor count:", len(feature_names))
print("Training records:", model_metadata.get("training_records"))
print("Testing records:", model_metadata.get("testing_records"))
print("Model reload verification: PASSED")

MODEL EXPORT SUCCESSFUL
Source: C:\Users\hp\Nassau_Candy_Final_Submission\models\Nassau_Lead_Time_Model.joblib
Output: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\models\Nassau_Lead_Time_Model.joblib
Model: Tuned Random Forest
Target: Lead_Time_Days
Predictor count: 20
Training records: 8132
Testing records: 2062
Model reload verification: PASSED


In [21]:
# Model Metrics Export 

# Model Metrics Export - preserve original metric files and document differences

import json
import shutil
import pandas as pd

METRICS_SOURCE_DIR = PROJECT_ROOT / "data" / "processed"
METRICS_OUTPUT_DIR = APP_REPORTS / "model_metrics"

METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metric_files = [
    "ML_Final_Model_Comparison.csv",
    "ML_Cross_Validation.csv",
    "ML_Predictions.csv",
    "ML_LeadTime_Ranges.csv",
    "ML_Dashboard_Metrics.csv"
]

metrics_manifest = []

for filename in metric_files:
    source = METRICS_SOURCE_DIR / filename

    if not source.exists():
        if filename == "ML_Dashboard_Metrics.csv":
            print("Optional metric file not found:", source)
            continue
        raise FileNotFoundError(f"Required metric file not found: {source}")

    df = pd.read_csv(source)
    output = METRICS_OUTPUT_DIR / filename

    shutil.copy2(source, output)

    reloaded = pd.read_csv(output)

    if df.shape != reloaded.shape:
        raise AssertionError(f"Shape mismatch after export: {filename}")

    metrics_manifest.append({
        "File": filename,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Export_Status": "PASSED"
    })

# Export the original model metadata
metadata_output = METRICS_OUTPUT_DIR / "model_metadata.json"
shutil.copy2(MODEL_METADATA_SOURCE, metadata_output)

# Record the known discrepancy without changing source values
metrics_review = {
    "review_status": "REVIEW_REQUIRED",
    "note": (
        "Saved model metadata test metrics differ from the Tuned Random Forest "
        "row in ML_Final_Model_Comparison.csv. Values are preserved separately; "
        "no reconciliation or replacement has been performed."
    ),
    "metadata_metrics": {
        "MAE": model_metadata.get("test_MAE"),
        "RMSE": model_metadata.get("test_RMSE"),
        "R2": model_metadata.get("test_R2")
    },
    "comparison_file": "ML_Final_Model_Comparison.csv",
    "comparison_values_for_tuned_random_forest": {
        "MAE": 139.670688,
        "RMSE": 163.675428,
        "R2": 0.619499
    },
    "cross_validation_note": (
        "ML_Cross_Validation.csv contains Random Forest, Gradient Boosting "
        "and Linear Regression; it does not list Tuned Random Forest."
    )
}

review_output = METRICS_OUTPUT_DIR / "metrics_review_note.json"

with open(review_output, "w", encoding="utf-8") as file:
    json.dump(metrics_review, file, indent=4)

manifest_df = pd.DataFrame(metrics_manifest)
manifest_output = METRICS_OUTPUT_DIR / "metrics_export_manifest.csv"
manifest_df.to_csv(manifest_output, index=False)

print("=" * 65)
print("MODEL METRICS EXPORT COMPLETED")
print("=" * 65)
print(manifest_df.to_string(index=False))
print("Metadata exported:", metadata_output)
print("Review note:", review_output)
print("Metric discrepancy: PRESERVED FOR REVIEW")

MODEL METRICS EXPORT COMPLETED
                         File  Rows  Columns Export_Status
ML_Final_Model_Comparison.csv     4        4        PASSED
      ML_Cross_Validation.csv     3        4        PASSED
           ML_Predictions.csv  2062        5        PASSED
       ML_LeadTime_Ranges.csv     5        5        PASSED
     ML_Dashboard_Metrics.csv     4        2        PASSED
Metadata exported: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\reports\model_metrics\model_metadata.json
Review note: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\reports\model_metrics\metrics_review_note.json
Metric discrepancy: PRESERVED FOR REVIEW


In [23]:
# Recommendation Export 

#  Recommendation Export - copy identified recommendation outputs

import shutil
import pandas as pd

RECOMMENDATION_SOURCE_DIR = (
    PROJECT_ROOT / "reports" / "optimization_recommendations"
)

RECOMMENDATION_OUTPUT_DIR = APP_REPORTS / "recommendations"
RECOMMENDATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_recommendation_files = [
    "Phase_10_Product_Review_Priorities.csv",
    "Phase_10_Factory_Option_Screening.csv",
    "task_10_04_lead_time_comparison.csv",
    "task_10_13_top_3_products.csv",
    "task_10_14_top_5_products.csv",
    "task_10_15_top_10_products.csv",
    "task_10_16_recommendation_explanations.csv",
    "task_10_17_coverage_kpis.csv",
    "task_10_18_unsupported_scenarios.csv",
    "task_10_22_business_action_plan.csv",
    "ranked_product_recommendations.csv"
]

recommendation_manifest = []

for filename in required_recommendation_files:
    source = RECOMMENDATION_SOURCE_DIR / filename

    if not source.exists():
        raise FileNotFoundError(
            f"Required Phase 10 recommendation file not found: {source}"
        )

    source_df = pd.read_csv(source)
    output = RECOMMENDATION_OUTPUT_DIR / filename

    shutil.copy2(source, output)

    exported_df = pd.read_csv(output)

    if source_df.shape != exported_df.shape:
        raise AssertionError(f"Export mismatch: {filename}")

    recommendation_manifest.append({
        "File": filename,
        "Rows": len(exported_df),
        "Columns": len(exported_df.columns),
        "Status": "PASSED"
    })

recommendation_manifest_df = pd.DataFrame(recommendation_manifest)

recommendation_manifest_df.to_csv(
    RECOMMENDATION_OUTPUT_DIR / "recommendation_export_manifest.csv",
    index=False
)

print("=" * 65)
print("RECOMMENDATION EXPORT COMPLETED")
print("=" * 65)
print(recommendation_manifest_df.to_string(index=False))
print(
    "\nQualification: Product review priorities are descriptive review scores. "
    "Alternative factory options remain unverified; factory-specific lead-time "
    "and cost benefits are not established."
)

RECOMMENDATION EXPORT COMPLETED
                                      File  Rows  Columns Status
    Phase_10_Product_Review_Priorities.csv    15       11 PASSED
     Phase_10_Factory_Option_Screening.csv    75       15 PASSED
       task_10_04_lead_time_comparison.csv    75       17 PASSED
             task_10_13_top_3_products.csv     3       13 PASSED
             task_10_14_top_5_products.csv     5       13 PASSED
            task_10_15_top_10_products.csv    10       13 PASSED
task_10_16_recommendation_explanations.csv    15       14 PASSED
              task_10_17_coverage_kpis.csv     7        4 PASSED
      task_10_18_unsupported_scenarios.csv    60       13 PASSED
       task_10_22_business_action_plan.csv     7        6 PASSED
        ranked_product_recommendations.csv    45       35 PASSED

Qualification: Product review priorities are descriptive review scores. Alternative factory options remain unverified; factory-specific lead-time and cost benefits are not established.


In [29]:
# KPI Export

#  KPI and Risk Export - copy all available KPI CSV outputs

import shutil
import pandas as pd

KPI_SOURCE_DIR = PROJECT_ROOT / "reports" / "kpi_risk_analysis"
KPI_OUTPUT_DIR = APP_DATA / "kpis"

if not KPI_SOURCE_DIR.exists():
    raise FileNotFoundError(f"KPI source folder not found: {KPI_SOURCE_DIR}")

KPI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

kpi_source_files = sorted(KPI_SOURCE_DIR.glob("*.csv"))

if not kpi_source_files:
    raise FileNotFoundError("No KPI CSV files found in Phase 11 output folder.")

kpi_manifest = []

for source in kpi_source_files:
    source_df = pd.read_csv(source)
    output = KPI_OUTPUT_DIR / source.name

    shutil.copy2(source, output)

    exported_df = pd.read_csv(output)

    if source_df.shape != exported_df.shape:
        raise AssertionError(f"KPI export mismatch: {source.name}")

    kpi_manifest.append({
        "File": source.name,
        "Rows": len(exported_df),
        "Columns": len(exported_df.columns),
        "Status": "PASSED"
    })

kpi_manifest_df = pd.DataFrame(kpi_manifest)

kpi_manifest_df.to_csv(
    KPI_OUTPUT_DIR / "kpi_export_manifest.csv",
    index=False
)

dashboard_file = KPI_OUTPUT_DIR / "kpi_dashboard_data.csv"
risk_file = KPI_OUTPUT_DIR / "risk_register.csv"

if not dashboard_file.exists():
    raise FileNotFoundError("kpi_dashboard_data.csv was not exported.")

if not risk_file.exists():
    raise FileNotFoundError("risk_register.csv was not exported.")

print("=" * 65)
print("KPI EXPORT COMPLETED")
print("=" * 65)
print("CSV files exported:", len(kpi_source_files))
print(kpi_manifest_df.to_string(index=False))
print("Dashboard data:", dashboard_file)
print("Risk register:", risk_file)

KPI EXPORT COMPLETED
CSV files exported: 19
                                 File  Rows  Columns Status
      high_derived_lead_time_risk.csv    15       12 PASSED
    kpi_average_derived_lead_time.csv     1        7 PASSED
               kpi_dashboard_data.csv     9        6 PASSED
kpi_estimated_lead_time_reduction.csv     1        7 PASSED
                 kpi_gross_profit.csv     1        7 PASSED
        kpi_lead_time_variability.csv     2        5 PASSED
     kpi_median_derived_lead_time.csv     1        7 PASSED
                kpi_profit_margin.csv     1        6 PASSED
                  kpi_total_sales.csv     1        7 PASSED
                  low_margin_risk.csv    15       12 PASSED
             low_sample_size_risk.csv    15        6 PASSED
         model_uncertainty_report.csv     5        4 PASSED
         phase11_integrity_report.csv     9        3 PASSED
           profit_stability_proxy.csv    15       12 PASSED
          recommendation_coverage.csv     7        6 PAS

In [31]:
# Reusable Prediction Functions

# Create reusable model prediction module

prediction_module_code = r'''
from pathlib import Path
import json
import joblib
import pandas as pd


def load_model(model_path):
    """Load a saved joblib model."""
    model_path = Path(model_path)

    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")

    return joblib.load(model_path)


def load_feature_schema(schema_path):
    """Load the saved model feature schema."""
    schema_path = Path(schema_path)

    if not schema_path.exists():
        raise FileNotFoundError(f"Feature schema not found: {schema_path}")

    with open(schema_path, "r", encoding="utf-8") as file:
        schema = json.load(file)

    features = schema.get("predictor_features", [])

    if not features:
        raise ValueError("Feature schema contains no predictor features.")

    return features


def validate_prediction_data(data, feature_names):
    """Validate required model input columns without inventing values."""
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Prediction input must be a pandas DataFrame.")

    missing = [col for col in feature_names if col not in data.columns]

    if missing:
        raise ValueError(f"Missing model features: {missing}")

    if data.empty:
        raise ValueError("Prediction input is empty.")

    return data.loc[:, feature_names].copy()


def predict_lead_time(model, data, feature_names):
    """Return model predictions in days; interpretation depends on target definition."""
    X = validate_prediction_data(data, feature_names)
    predictions = model.predict(X)

    return pd.Series(
        predictions,
        index=data.index,
        name="Predicted_Lead_Time_Days"
    )
'''

prediction_module_path = APP_SRC / "prediction_utils.py"
prediction_module_path.write_text(
    prediction_module_code,
    encoding="utf-8"
)

print("Prediction module created:", prediction_module_path)
print("Functions: load_model, load_feature_schema, validate_prediction_data, predict_lead_time")

Prediction module created: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\src\prediction_utils.py
Functions: load_model, load_feature_schema, validate_prediction_data, predict_lead_time


In [33]:
# Feature Transformation Functions

# Create reusable feature transformation module

feature_module_code = r'''
import numpy as np
import pandas as pd


def add_order_date_features(data, date_column="Order Date"):
    """
    Add date features used by the saved model.
    Original date column is preserved.
    """
    if date_column not in data.columns:
        raise ValueError(f"Missing date column: {date_column}")

    result = data.copy()

    parsed_dates = pd.to_datetime(
        result[date_column],
        dayfirst=True,
        errors="coerce"
    )

    if parsed_dates.isna().any():
        raise ValueError(
            "Some order dates could not be parsed. "
            "Review source date values before prediction."
        )

    result["Order_Year"] = parsed_dates.dt.year
    result["Order_Month"] = parsed_dates.dt.month
    result["Order_Quarter"] = parsed_dates.dt.quarter
    result["Order_Weekday"] = parsed_dates.dt.dayofweek

    return result


def safe_divide(numerator, denominator):
    """Divide safely; zero denominators produce NaN."""
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")

    result = numerator.div(denominator.replace(0, np.nan))
    return result.replace([np.inf, -np.inf], np.nan)


def add_financial_features(data):
    """
    Add financial ratio features using the source dataset's
    Sales, Cost, Gross Profit and Units fields.

    Profit_Margin_Pct is expressed as a percentage.
    """
    required = ["Sales", "Cost", "Gross Profit", "Units"]
    missing = [col for col in required if col not in data.columns]

    if missing:
        raise ValueError(f"Missing financial columns: {missing}")

    result = data.copy()

    result["Profit_Margin_Pct"] = (
        safe_divide(result["Gross Profit"], result["Sales"]) * 100
    )

    result["Sales_Per_Unit"] = safe_divide(
        result["Sales"], result["Units"]
    )

    result["Cost_Per_Unit"] = safe_divide(
        result["Cost"], result["Units"]
    )

    result["Profit_Per_Unit"] = safe_divide(
        result["Gross Profit"], result["Units"]
    )

    return result


def prepare_model_features(data):
    """
    Apply date and financial transformations.
    Does not remove rows or overwrite original source columns.
    """
    result = add_order_date_features(data)
    result = add_financial_features(result)
    return result
'''

feature_module_path = APP_SRC / "feature_utils.py"
feature_module_path.write_text(
    feature_module_code,
    encoding="utf-8"
)

print("Feature module created:", feature_module_path)
print("Functions: add_order_date_features, safe_divide, add_financial_features, prepare_model_features")
print("Original columns are preserved.")

Feature module created: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\src\feature_utils.py
Functions: add_order_date_features, safe_divide, add_financial_features, prepare_model_features
Original columns are preserved.


In [35]:
# Scenario Functions

# Create reusable scenario support module

scenario_module_code = r'''
import pandas as pd


def assess_factory_option_support(option_row):
    """
    Assess whether a factory option has evidence for factory-specific
    lead-time and cost estimation.
    """
    lead_time_supported = bool(
        option_row.get("Factory_Specific_Lead_Time_Estimable", False)
    )

    cost_supported = bool(
        option_row.get("Factory_Specific_Cost_Available", False)
    )

    capacity_supported = bool(
        option_row.get("Capacity_Data_Available", False)
    )

    if lead_time_supported and cost_supported and capacity_supported:
        status = "REQUIRES FURTHER VALIDATION"
    else:
        status = "INSUFFICIENT FACTORY-SPECIFIC EVIDENCE"

    return {
        "Lead_Time_Evidence_Available": lead_time_supported,
        "Cost_Evidence_Available": cost_supported,
        "Capacity_Evidence_Available": capacity_supported,
        "Scenario_Support_Status": status,
        "Estimated_Lead_Time_Change_Days": None,
        "Estimated_Cost_Saving": None,
        "Success_Probability": None
    }


def review_factory_options(option_data):
    """Add evidence status without changing source scores or estimates."""
    required = [
        "Factory_Specific_Lead_Time_Estimable",
        "Factory_Specific_Cost_Available",
        "Capacity_Data_Available"
    ]

    missing = [col for col in required if col not in option_data.columns]

    if missing:
        raise ValueError(f"Missing scenario evidence columns: {missing}")

    result = option_data.copy()

    statuses = result.apply(
        lambda row: assess_factory_option_support(row)[
            "Scenario_Support_Status"
        ],
        axis=1
    )

    result["App_Scenario_Support_Status"] = statuses

    return result
'''

scenario_module_path = APP_SRC / "scenario_utils.py"
scenario_module_path.write_text(
    scenario_module_code,
    encoding="utf-8"
)

print("Scenario module created:", scenario_module_path)
print("Functions: assess_factory_option_support, review_factory_options")
print("No unsupported benefit or success-probability calculations are included.")

Scenario module created: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\src\scenario_utils.py
Functions: assess_factory_option_support, review_factory_options
No unsupported benefit or success-probability calculations are included.


In [37]:
# Recommendation Functions 

# Create reusable recommendation module

recommendation_module_code = r'''
import pandas as pd


def rank_existing_priorities(
    data,
    score_column="Balanced_Score",
    top_n=5
):
    """
    Rank existing product review scores.
    This is not a factory-feasibility or factory-benefit ranking.
    """
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if score_column not in data.columns:
        raise ValueError(f"Score column not found: {score_column}")

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    result = data.copy()
    result[score_column] = pd.to_numeric(
        result[score_column], errors="coerce"
    )

    result = result.dropna(subset=[score_column])

    result = result.sort_values(
        by=score_column,
        ascending=False,
        kind="mergesort"
    )

    result = result.head(top_n).reset_index(drop=True)
    result.insert(0, "App_Review_Priority_Rank", range(1, len(result) + 1))

    result["App_Ranking_Interpretation"] = (
        "Existing product review priority; not factory benefit or feasibility"
    )

    return result


def get_factory_option_warnings(option_data):
    """Return factory options that must remain flagged as unverified."""
    required = [
        "Product Name",
        "Reference Factory",
        "Factory Option",
        "Compatibility_Status",
        "Recommendation_Status"
    ]

    missing = [col for col in required if col not in option_data.columns]

    if missing:
        raise ValueError(f"Missing factory option columns: {missing}")

    result = option_data.copy()

    if "Is_Reference_Mapping" in result.columns:
        result["App_Option_Interpretation"] = result[
            "Is_Reference_Mapping"
        ].map({
            True: "Reference mapping only; actual dispatch not verified",
            False: "Unverified alternative; do not treat as approved"
        })

    return result
'''

recommendation_module_path = APP_SRC / "recommendation_utils.py"
recommendation_module_path.write_text(
    recommendation_module_code,
    encoding="utf-8"
)

print("Recommendation module created:", recommendation_module_path)
print("Functions: rank_existing_priorities, get_factory_option_warnings")

Recommendation module created: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\src\recommendation_utils.py
Functions: rank_existing_priorities, get_factory_option_warnings


In [41]:
# App Data Loading and Integrity Test

# Final App Data Loading and Integrity Validation

from pathlib import Path
import hashlib
import json
import subprocess
import sys
import pandas as pd

# Expected original source
RAW_SOURCE = PROJECT_ROOT / "data" / "raw" / "Nassau Candy Distributor.csv"

EXPECTED_RAW_SHA256 = (
    "6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f"
)

# Expected exported files
required_files = [
    APP_DATA / "processed_dataset.csv",
    APP_DATA / "factory_master.csv",
    APP_DATA / "product_factory_mapping.csv",
    APP_MODELS / "Nassau_Lead_Time_Model.joblib",
    APP_MODELS / "model_metadata.json",
    APP_MODELS / "feature_schema.json",
    APP_REPORTS / "model_metrics" / "metrics_export_manifest.csv",
    APP_REPORTS / "recommendations" / "recommendation_export_manifest.csv",
    APP_DATA / "kpis" / "kpi_dashboard_data.csv",
    APP_DATA / "kpis" / "risk_register.csv",
    APP_SRC / "prediction_utils.py",
    APP_SRC / "feature_utils.py",
    APP_SRC / "scenario_utils.py",
    APP_SRC / "recommendation_utils.py"
]

validation_results = []

# Check file existence
for path in required_files:
    passed = path.exists() and path.is_file()

    validation_results.append({
        "Check": f"File exists: {path.name}",
        "Passed": passed
    })

    if not passed:
        raise FileNotFoundError(f"Required app file missing: {path}")

# Reload app-facing data
processed_app = pd.read_csv(APP_DATA / "processed_dataset.csv")
factory_app = pd.read_csv(APP_DATA / "factory_master.csv")
mapping_app = pd.read_csv(APP_DATA / "product_factory_mapping.csv")
kpi_app = pd.read_csv(APP_DATA / "kpis" / "kpi_dashboard_data.csv")
risk_app = pd.read_csv(APP_DATA / "kpis" / "risk_register.csv")

# Validate expected source dimensions
raw_df = pd.read_csv(RAW_SOURCE)

assert raw_df.shape == (10194, 18), (
    f"Unexpected raw source shape: {raw_df.shape}"
)

assert len(processed_app) == 10194, (
    f"Unexpected processed export row count: {len(processed_app)}"
)

assert len(factory_app) == 5, (
    f"Expected 5 factories, found {len(factory_app)}"
)

assert len(mapping_app) == 15, (
    f"Expected 15 product mappings, found {len(mapping_app)}"
)

assert "kpi_dashboard_data.csv" in [
    p.name for p in (APP_DATA / "kpis").glob("*.csv")
]

assert not risk_app.empty, "Risk register is empty."
assert not kpi_app.empty, "KPI dashboard data is empty."

validation_results.extend([
    {"Check": "Original raw shape is 10,194 x 18", "Passed": True},
    {"Check": "Processed export contains 10,194 rows", "Passed": True},
    {"Check": "Factory Master contains 5 records", "Passed": True},
    {"Check": "Product mapping contains 15 records", "Passed": True},
    {"Check": "KPI dashboard data reloads", "Passed": True},
    {"Check": "Risk register reloads", "Passed": True}
])

# Verify every mapped factory exists in Factory Master
unknown_factory_names = (
    set(mapping_app["Mapped Factory"].astype(str))
    - set(factory_app["Factory_Name"].astype(str))
)

assert not unknown_factory_names, (
    f"Unknown mapped factories: {unknown_factory_names}"
)

validation_results.append({
    "Check": "All mapped factories exist in Factory Master",
    "Passed": True
})

# Verify original source hash remains unchanged
sha256 = hashlib.sha256()

with open(RAW_SOURCE, "rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_raw_hash = sha256.hexdigest()

assert actual_raw_hash == EXPECTED_RAW_SHA256, (
    "Original source hash changed. Stop and investigate."
)

validation_results.append({
    "Check": "Original CSV SHA256 unchanged",
    "Passed": True
})

# Fresh Python process test: model loading and module imports
fresh_test_code = f"""
import sys
from pathlib import Path
import joblib
import json

app_root = Path({str(APP_ROOT)!r})
app_src = Path({str(APP_SRC)!r})
model_path = Path({str(APP_MODELS / "Nassau_Lead_Time_Model.joblib")!r})
schema_path = Path({str(APP_MODELS / "feature_schema.json")!r})

sys.path.insert(0, str(app_src))

model = joblib.load(model_path)

with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

assert model is not None
assert len(schema["predictor_features"]) > 0

import prediction_utils
import feature_utils
import scenario_utils
import recommendation_utils

print("Fresh-process model load: PASSED")
print("Fresh-process module imports: PASSED")
print("Feature count:", len(schema["predictor_features"]))
"""

fresh_test = subprocess.run(
    [sys.executable, "-c", fresh_test_code],
    capture_output=True,
    text=True,
    cwd=str(APP_ROOT)
)

if fresh_test.returncode != 0:
    raise RuntimeError(
        "Fresh Python process validation failed:\n"
        + fresh_test.stdout
        + "\n"
        + fresh_test.stderr
    )

validation_results.append({
    "Check": "Model loads in fresh Python process",
    "Passed": True
})

validation_results.append({
    "Check": "All four app modules import in fresh process",
    "Passed": True
})

# Display final validation report
validation_df = pd.DataFrame(validation_results)

validation_output = APP_REPORTS / "validation_report.csv"
validation_df.to_csv(validation_output, index=False)

print("=" * 70)
print(" APP DATA VALIDATION")
print("=" * 70)

print(validation_df.to_string(index=False))

print("\nTotal checks:", len(validation_df))
print("Passed:", int(validation_df["Passed"].sum()))
print("Failed:", int((~validation_df["Passed"]).sum()))

print("\nOriginal source SHA256:", actual_raw_hash)
print("Fresh-process test output:")
print(fresh_test.stdout)

print("Validation report:", validation_output)

if not validation_df["Passed"].all():
    raise RuntimeError("validation failed.")

print("\n VALIDATION: PASSED")

 APP DATA VALIDATION
                                          Check  Passed
             File exists: processed_dataset.csv    True
                File exists: factory_master.csv    True
       File exists: product_factory_mapping.csv    True
     File exists: Nassau_Lead_Time_Model.joblib    True
               File exists: model_metadata.json    True
               File exists: feature_schema.json    True
       File exists: metrics_export_manifest.csv    True
File exists: recommendation_export_manifest.csv    True
            File exists: kpi_dashboard_data.csv    True
                 File exists: risk_register.csv    True
               File exists: prediction_utils.py    True
                  File exists: feature_utils.py    True
                 File exists: scenario_utils.py    True
           File exists: recommendation_utils.py    True
              Original raw shape is 10,194 x 18    True
          Processed export contains 10,194 rows    True
              Factory Maste